# 04 — Sepsis Supplementary Analysis

Full learner grid, oracle-selected estimator results, ranking metrics,
random partition analysis, and detailed support/overlap tables.

These results belong in the supplementary material of the paper.

In [1]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'sepsis'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    build_full_learner_table, all_metrics, ranking_metrics_within_groups, ranking_comparison_table,
    resolve_alternative, describe_paired_test,
    METHODS_ORDER, LEARNERS_ORDER
)
from helpers.plotting import METHOD_LABELS

results_by_seed = load_checkpoint(CFG.CHECKPOINT_PATH)
completed_seeds = sorted(results_by_seed.keys())
print(f'Loaded {len(completed_seeds)} seeds.')


Loaded 20 seeds.


## 1. Full Learner Grid — ATE MSE

In [2]:
ate_grid = build_full_learner_table(results_by_seed, metric='ate_mse')
print('ATE MSE: mean across seeds (rows=learners, cols=methods)')
display(ate_grid.round(5))
ate_grid.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_ate_mse.csv'))

ATE MSE: mean across seeds (rows=learners, cols=methods)


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,3.384863e+06,4.853166e+06,1.257100e+06,5.533722e+06,2.660277e+06,8.509998e+28
GLOBAL_MISSINGNESS,2.199108e+06,4.809813e+06,1.311348e+06,5.557563e+06,2.647049e+06,5.532328e+27
GLOBAL_MISSINGNESS_CDV,1.651083e+06,4.828264e+06,2.013636e+06,5.770857e+06,2.473366e+06,3.344952e+27
MATCHED_RANDOM_PARTITIONS,2.261983e+06,4.906495e+06,1.385564e+06,5.617376e+06,2.661331e+06,5.532328e+27
CDV_SEPARATE,4.039395e+06,4.649796e+06,1.148494e+06,5.397495e+06,2.912902e+06,8.509998e+28


In [3]:
baseline_methods = [m for m in METHODS_ORDER if m != 'CDV_SEPARATE']
cdv_ate_row = ate_grid.loc['CDV_SEPARATE']
ate_improvement = (ate_grid.loc[baseline_methods] - cdv_ate_row) / ate_grid.loc[baseline_methods] * 100
ate_improvement.index = [METHOD_LABELS.get(m, m) for m in ate_improvement.index]
print('CDV_SEPARATE improvement (%) over each method, per learner (ATE MSE):')
display(ate_improvement.round(2))
ate_improvement.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_ate_mse_improvement.csv'))

CDV_SEPARATE improvement (%) over each method, per learner (ATE MSE):


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
Global (Sentinel),-19.34,4.19,8.64,2.46,-9.50,0.00
Global + Missingness,-83.68,3.33,12.42,2.88,-10.04,-1438.23
Global + Miss. + CDV ID,-144.65,3.70,42.96,6.47,-17.77,-2444.13
Matched Random Partitions,-78.58,5.23,17.11,3.91,-9.45,-1438.23


## 2. Full Learner Grid — CATE MSE

In [4]:
cate_grid = build_full_learner_table(results_by_seed, metric='cate_mse')
print('CATE MSE: mean across seeds')
display(cate_grid.round(5))
cate_grid.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_cate_mse.csv'))

CATE MSE: mean across seeds


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
GLOBAL_SENTINEL,4.395847e+07,6.243208e+06,2.026219e+06,2.567744e+07,1.664515e+07,2.067930e+31
GLOBAL_MISSINGNESS,2.603699e+08,6.153419e+06,2.080467e+06,2.549487e+07,1.639208e+07,1.344356e+30
GLOBAL_MISSINGNESS_CDV,1.864362e+08,6.176796e+06,8.229911e+06,2.650980e+07,1.640271e+07,8.128233e+29
MATCHED_RANDOM_PARTITIONS,2.603371e+08,5.956461e+06,2.172590e+06,2.339411e+07,1.454710e+07,1.344356e+30
CDV_SEPARATE,4.399867e+07,6.027037e+06,1.968310e+06,2.555455e+07,1.648137e+07,2.067930e+31


In [5]:
cdv_cate_row = cate_grid.loc['CDV_SEPARATE']
cate_improvement = (cate_grid.loc[baseline_methods] - cdv_cate_row) / cate_grid.loc[baseline_methods] * 100
cate_improvement.index = [METHOD_LABELS.get(m, m) for m in cate_improvement.index]
print('CDV_SEPARATE improvement (%) over each method, per learner (CATE MSE):')
display(cate_improvement.round(2))
cate_improvement.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'full_learner_cate_mse_improvement.csv'))

CDV_SEPARATE improvement (%) over each method, per learner (CATE MSE):


,DR_RF,S_RF,S_Linear,T_RF,X_RF,Double_ML
Global (Sentinel),-0.09,3.46,2.86,0.48,0.98,0.00
Global + Missingness,83.10,2.05,5.39,-0.23,-0.54,-1438.23
Global + Miss. + CDV ID,76.40,2.42,76.08,3.60,-0.48,-2444.13
Matched Random Partitions,83.10,-1.18,9.40,-9.23,-13.30,-1438.23


## 3. Ranking Metrics — Kendall τ and Spearman ρ (DR-RF)

In [6]:
print(f'Ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds')
ranking_rows = []
for method in METHODS_ORDER:
    tau_vals, rho_vals = [], []
    for sr in results_by_seed.values():
        m = sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {})
        tau_vals.append(m.get('kendall_tau', np.nan))
        rho_vals.append(m.get('spearman_rho', np.nan))
    ranking_rows.append({
        'Method': METHOD_LABELS.get(method, method),
        'Kendall τ mean': np.nanmean(tau_vals),
        'Kendall τ std':  np.nanstd(tau_vals),
        'Spearman ρ mean': np.nanmean(rho_vals),
        'Spearman ρ std':  np.nanstd(rho_vals),
        'N seeds': int(np.sum(np.isfinite(tau_vals))),
    })
ranking_df = pd.DataFrame(ranking_rows)
display(ranking_df.round(4))
ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics.csv'), index=False)

Ranking metrics (DR_RF) — mean ± std across seeds


,Method,Kendall τ mean,Kendall τ std,Spearman ρ mean,Spearman ρ std,N seeds
0,Global (Sentinel),0.1161,0.1186,0.1692,0.1622,20
1,Global + Missingness,0.1151,0.1194,0.1645,0.1656,20
2,Global + Miss. + CDV ID,0.1145,0.1053,0.1635,0.1444,20
3,Matched Random Partitions,0.1263,0.1251,0.1796,0.1731,20
4,CDV Separate (proposed),0.1504,0.1547,0.2113,0.2111,20


In [7]:
selected_learner_rank = CFG.PRIMARY_LEARNER  # change to compare a different learner

print(f'CDV_SEPARATE vs other methods — Kendall τ ({selected_learner_rank})')
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
tau_cmp = ranking_comparison_table(results_by_seed, selected_learner_rank, metric='kendall_tau',
                                    ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
tau_cmp.index = [METHOD_LABELS.get(m, m) for m in tau_cmp.index]

# Add CDV row at top
cdv_tau_vals = np.array([
    sr.get("metrics", {}).get("CDV_SEPARATE", {}).get(selected_learner_rank, {}).get('kendall_tau', np.nan)
    for sr in results_by_seed.values()
])
n_seeds_tau = int(np.sum(np.isfinite(cdv_tau_vals)))
cdv_tau_row = pd.DataFrame([{
    'method': 'CDV_SEPARATE',
    'mean': float(np.nanmean(cdv_tau_vals)),
    'std': float(np.nanstd(cdv_tau_vals)),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
    'N seeds': n_seeds_tau
}])
tau_cmp = tau_cmp.reset_index().rename(columns={'index': 'method'})
tau_cmp = pd.concat([cdv_tau_row, tau_cmp], ignore_index=True).set_index('method')
display(tau_cmp.round(4))

print(f'\nCDV_SEPARATE vs other methods — Spearman ρ ({selected_learner_rank})')
print(describe_paired_test('CDV_SEPARATE', 'method', 'Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
rho_cmp = ranking_comparison_table(results_by_seed, selected_learner_rank, metric='spearman_rho',
                                    ci_method=CFG.CI_METHOD, sided=CFG.CI_SIDED)
rho_cmp.index = [METHOD_LABELS.get(m, m) for m in rho_cmp.index]

# Add CDV row at top
cdv_rho_vals = np.array([
    sr.get("metrics", {}).get("CDV_SEPARATE", {}).get(selected_learner_rank, {}).get('spearman_rho', np.nan)
    for sr in results_by_seed.values()
])
n_seeds_rho = int(np.sum(np.isfinite(cdv_rho_vals)))
cdv_rho_row = pd.DataFrame([{
    'method': 'CDV_SEPARATE',
    'mean': float(np.nanmean(cdv_rho_vals)),
    'std': float(np.nanstd(cdv_rho_vals)),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
    'improvement_pct': np.nan,
}])
rho_cmp = rho_cmp.reset_index().rename(columns={'index': 'method'})
rho_cmp = pd.concat([cdv_rho_row, rho_cmp], ignore_index=True).set_index('method')
display(rho_cmp.round(4))

tau_cmp.to_csv(os.path.join(CFG.ARTIFACTS_DIR, f'ranking_metrics_tau_{selected_learner_rank}.csv'), index=False)
rho_cmp.to_csv(os.path.join(CFG.ARTIFACTS_DIR, f'ranking_metrics_rho_{selected_learner_rank}.csv'), index=False)


CDV_SEPARATE vs other methods — Kendall τ (DR_RF)
paired bootstrap CI (10,000 resamples), two-sided (alpha=0.05). Δ = CDV_SEPARATE − method  (Kendall τ), paired per outer seed. H1: Δ ≠ 0.


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),N seeds,improvement_pct
method,,,,,,
CDV_SEPARATE,0.1504,0.1547,-,-,20.0,NaN
Global (Sentinel),0.1161,0.1186,"[0.0029, 0.0689]",0.0294,NaN,29.5021
Global + Missingness,0.1151,0.1194,"[-0.0002, 0.0714]",0.0518,NaN,30.6231
Global + Miss. + CDV ID,0.1145,0.1053,"[-0.0008, 0.0740]",0.0572,NaN,31.3644
Matched Random Partitions,0.1263,0.1251,"[-0.0061, 0.0576]",0.1322,NaN,19.0960



CDV_SEPARATE vs other methods — Spearman ρ (DR_RF)
paired bootstrap CI (10,000 resamples), two-sided (alpha=0.05). Δ = CDV_SEPARATE − method  (Spearman ρ), paired per outer seed. H1: Δ ≠ 0.


,mean,std,paired 95% CI of Δ (CDV_SEPARATE − method),p_value (paired),improvement_pct
method,,,,,
CDV_SEPARATE,0.2113,0.2111,-,-,NaN
Global (Sentinel),0.1692,0.1622,"[0.0007, 0.0868]",0.0456,24.8270
Global + Missingness,0.1645,0.1656,"[-0.0009, 0.0947]",0.055,28.4570
Global + Miss. + CDV ID,0.1635,0.1444,"[-0.0021, 0.0991]",0.0626,29.2079
Matched Random Partitions,0.1796,0.1731,"[-0.0094, 0.0764]",0.146,17.6218


## 4. Within-CDV Ranking Metrics — Kendall τ and Spearman ρ (primary learner)

Section 3's metrics pool predictions across all CDV groups (and OTHER) before ranking.
Here, Kendall τ / Spearman ρ are computed separately WITHIN each CDV group, then
size-weighted averaged across groups — isolating within-subgroup ranking quality from
across-group ordering effects.


In [8]:
print(f'Within-CDV ranking metrics ({CFG.PRIMARY_LEARNER}) — mean ± std across seeds')
within_rows = []
for method in METHODS_ORDER:
    tau_vals, rho_vals = [], []
    for sr in results_by_seed.values():
        pred = sr.get('predictions', {}).get(method, {}).get(CFG.PRIMARY_LEARNER, {})
        ite_pred = np.asarray(pred.get('ite_pred', []))
        ite_true = np.asarray(pred.get('ite_true', []))
        variant = np.asarray(pred.get('variant', []))
        if len(ite_pred) == 0:
            continue
        m = ranking_metrics_within_groups(ite_pred, ite_true, variant)
        tau_vals.append(m['kendall_tau_within'])
        rho_vals.append(m['spearman_rho_within'])
    within_rows.append({
        'Method': METHOD_LABELS.get(method, method),
        'Kendall τ (within-CDV) mean': np.nanmean(tau_vals),
        'Kendall τ (within-CDV) std':  np.nanstd(tau_vals),
        'Spearman ρ (within-CDV) mean': np.nanmean(rho_vals),
        'Spearman ρ (within-CDV) std':  np.nanstd(rho_vals),
        'N seeds': int(np.sum(np.isfinite(tau_vals))),
    })

within_ranking_df = pd.DataFrame(within_rows)
display(within_ranking_df.round(4))
within_ranking_df.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'ranking_metrics_within_cdv.csv'), index=False)


Within-CDV ranking metrics (DR_RF) — mean ± std across seeds


,Method,Kendall τ (within-CDV) mean,Kendall τ (within-CDV) std,Spearman ρ (within-CDV) mean,Spearman ρ (within-CDV) std,N seeds
0,Global (Sentinel),0.1067,0.1150,0.1567,0.1561,20
1,Global + Missingness,0.0964,0.1380,0.1382,0.1924,20
2,Global + Miss. + CDV ID,0.0970,0.1274,0.1392,0.1778,20
3,Matched Random Partitions,0.1104,0.1389,0.1574,0.1929,20
4,CDV Separate (proposed),0.1446,0.1524,0.2035,0.2080,20


## 5. Oracle-Selected Estimator Results

In [9]:
oracle_rows = []
for seed, sr in results_by_seed.items():
    for method in METHODS_ORDER:
        oracle = sr.get('oracle', {}).get(method, {})
        m = oracle.get('metrics', {})
        oracle_rows.append({
            'outer_seed':       seed,
            'method':           method,
            'selected_learner': oracle.get('selected_learner', 'N/A'),
            'ate_mse':          m.get('ate_mse', np.nan),
            'cate_mse':         m.get('cate_mse', np.nan),
            'kendall_tau':      m.get('kendall_tau', np.nan),
            'spearman_rho':     m.get('spearman_rho', np.nan),
        })

oracle_df = pd.DataFrame(oracle_rows)
print('Oracle results by method:')
display(oracle_df.groupby('method')[['ate_mse', 'cate_mse']].mean().round(5))

print('\nLearner selection frequency:')
display(oracle_df.groupby(['method', 'selected_learner']).size().unstack(fill_value=0))

oracle_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'oracle_results.parquet'), index=False)
print(f'Saved: oracle_results.parquet')

Oracle results by method:


,ate_mse,cate_mse
method,,
CDV_SEPARATE,4.424557e+06,5.959357e+06
GLOBAL_MISSINGNESS,4.424557e+06,5.959357e+06
GLOBAL_MISSINGNESS_CDV,4.451544e+06,6.060743e+06
GLOBAL_SENTINEL,4.468477e+06,6.109646e+06
MATCHED_RANDOM_PARTITIONS,4.424557e+06,5.959357e+06



Learner selection frequency:


selected_learner,S_RF,X_RF
method,,
CDV_SEPARATE,18,2
GLOBAL_MISSINGNESS,18,2
GLOBAL_MISSINGNESS_CDV,18,2
GLOBAL_SENTINEL,18,2
MATCHED_RANDOM_PARTITIONS,18,2


Saved: oracle_results.parquet


## 6. Random Partition Analysis

In [10]:
# Show that MATCHED_RANDOM_PARTITIONS results are based on averaged permutations
rp_rows = []
for seed, sr in results_by_seed.items():
    rp_preds = sr.get('predictions', {}).get('MATCHED_RANDOM_PARTITIONS', {})
    for lname, pred in rp_preds.items():
        n_perms = len(pred.get('perm_ite_preds', []))
        rp_rows.append({'outer_seed': seed, 'learner': lname, 'n_permutations': n_perms})

if rp_rows:
    rp_df = pd.DataFrame(rp_rows)
    print('Permutations per seed × learner (should all be N_RANDOM_PERMUTATIONS):')
    display(rp_df.groupby('learner')['n_permutations'].describe())
else:
    print('No random partition data found.')

Permutations per seed × learner (should all be N_RANDOM_PERMUTATIONS):


,count,mean,std,min,25%,50%,75%,max
learner,,,,,,,,
DR_RF,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0
Double_ML,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0
S_Linear,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0
S_RF,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0
T_RF,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0
X_RF,20.0,15.0,0.0,15.0,15.0,15.0,15.0,15.0


## 7. Build Tidy Main Results DataFrame

In [11]:
main_rows = []
for seed, sr in results_by_seed.items():
    for method in METHODS_ORDER:
        for learner in LEARNERS_ORDER:
            m = sr.get('metrics', {}).get(method, {}).get(learner, {})
            main_rows.append({
                'dataset':           'sepsis',
                'outer_seed':        seed,
                'alpha':             np.nan,
                'method':            method,
                'learner':           learner,
                'ate_mse':           m.get('ate_mse', np.nan),
                'cate_mse':          m.get('cate_mse', np.nan),
                'kendall_tau':       m.get('kendall_tau', np.nan),
                'spearman_rho':      m.get('spearman_rho', np.nan),
                'n_train':           sr.get('n_train', np.nan),
                'n_test':            sr.get('n_test', np.nan),
                'n_retained_cdvs':   len(sr.get('retained_cdv_info', {})),
                'pct_other_train':   sr.get('pct_other_train', np.nan),
                'pct_other_test':    sr.get('pct_other_test', np.nan),
            })

main_df = pd.DataFrame(main_rows)
main_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'main_results.parquet'), index=False)
print(f'Tidy results DataFrame: {main_df.shape}')
print(f'Saved: {CFG.ARTIFACTS_DIR}/main_results.parquet')
display(main_df[main_df['learner'] == 'DR_RF'].groupby('method')[['ate_mse', 'cate_mse']].mean().round(5))

Tidy results DataFrame: (600, 14)
Saved: revised_experiment/sepsis/artifacts/main_results.parquet


,ate_mse,cate_mse
method,,
CDV_SEPARATE,4.039395e+06,4.399867e+07
GLOBAL_MISSINGNESS,2.199108e+06,2.603699e+08
GLOBAL_MISSINGNESS_CDV,1.651083e+06,1.864362e+08
GLOBAL_SENTINEL,3.384863e+06,4.395847e+07
MATCHED_RANDOM_PARTITIONS,2.261983e+06,2.603371e+08
